# 05 - Bowling Analysis

In [ ]:
import pandas as pd
import numpy as np

matches = pd.read_csv('data/matches_clean.csv')
deliveries = pd.read_csv('data/deliveries_clean.csv')
print(matches.shape, deliveries.shape)

## Setup
- Bowler wickets: exclude run-outs, retired hurt, obstructing the field (not credited to bowler)
- Runs conceded: exclude byes and leg byes

In [ ]:
not_bowler_wkt = ['run out', 'retired hurt', 'retired out', 'obstructing the field', 'None']
bowler_wkts = deliveries[(deliveries['is_wicket'] == 1) & (~deliveries['dismissal_kind'].isin(not_bowler_wkt))]

conceded = deliveries[~deliveries['extras_type'].isin(['byes', 'legbyes'])]
legal = deliveries[~deliveries['extras_type'].isin(['wides', 'noballs'])]

## Top wicket takers

In [ ]:
wickets = bowler_wkts.groupby('bowler').size().sort_values(ascending=False)
wickets.head(10)

## Economy rate (min 500 balls)

In [ ]:
bowl = pd.DataFrame({
    'wickets': wickets,
    'runs_conceded': conceded.groupby('bowler')['total_runs'].sum(),
    'balls': legal.groupby('bowler').size()
}).fillna(0)
bowl['economy'] = (bowl['runs_conceded'] / (bowl['balls'] / 6)).round(2)
bowl[bowl['balls'] >= 500].sort_values('economy').head(10)

## Bowling average & strike rate

In [ ]:
bowl['average'] = (bowl['runs_conceded'] / bowl['wickets']).replace([np.inf], np.nan).round(2)
bowl['strike_rate'] = (bowl['balls'] / bowl['wickets']).replace([np.inf], np.nan).round(2)
bowl[bowl['wickets'] >= 50].sort_values('average').head(10)

## Dismissal types

In [ ]:
deliveries[deliveries['dismissal_kind'] != 'None']['dismissal_kind'].value_counts()

## Dot balls

In [ ]:
dots = legal[legal['total_runs'] == 0].groupby('bowler').size().sort_values(ascending=False)
dots.head(10)

## Purple cap (most wickets per season)

In [ ]:
w = bowler_wkts.merge(matches[['id', 'season']], left_on='match_id', right_on='id')
sw = w.groupby(['season', 'bowler']).size().reset_index(name='wickets')
sw.loc[sw.groupby('season')['wickets'].idxmax()]

## Phase-wise bowling (overs are 0-indexed: 0-5 powerplay, 6-14 middle, 15-19 death)

In [ ]:
deliveries['phase'] = pd.cut(deliveries['over'], bins=[-1, 5, 14, 19], labels=['Powerplay', 'Middle', 'Death'])
phase = deliveries.groupby('phase', observed=True).agg(runs=('total_runs', 'sum'), balls=('ball', 'count'), wickets=('is_wicket', 'sum'))
phase['run_rate'] = (phase['runs'] / (phase['balls'] / 6)).round(2)
phase

## Best death-over bowlers (min 300 balls)

In [ ]:
death = deliveries[(deliveries['over'] >= 15) & (~deliveries['extras_type'].isin(['byes', 'legbyes']))]
dg = death.groupby('bowler').agg(runs=('total_runs', 'sum'), balls=('ball', 'count'))
dg['economy'] = (dg['runs'] / (dg['balls'] / 6)).round(2)
dg[dg['balls'] >= 300].sort_values('economy').head(10)